# Deriving envelopes & relaxations with SymPy

`discopt` can *derive* tight convex/concave envelopes for nonlinear atoms
symbolically, **verify** their soundness, and **code-generate** pure-JAX
relaxation closures that plug into the branch-and-bound solver. This extends the
hand-written McCormick library {cite:p}`McCormick1976,Tsoukalas2014` to terms
arising in gas-network, electrical-grid, and chemical-engineering optimization.

The toolkit lives in `discopt._jax.symbolic` and is a **design-time** dependency
(`pip install discopt[sympy]`): SymPy derives the formulas, but the solver hot
path runs only the generated JAX. Soundness — `cv(x) <= f(x) <= cc(x)` with `cv`
convex and `cc` concave — is the non-negotiable invariant, checked for every atom.


## 1. Deriving a univariate envelope

For a function `f(x)` on a box `[a, b]`, the engine classifies the curvature and
derives the envelope. For an odd cubic `x^3` (concave then convex through an
inflection at 0) it finds the convex-underestimator tangent point in closed form.


In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import sympy as sp
from discopt._jax.symbolic import derive_envelope, lambdify_envelope, verify_envelope

x = sp.Symbol('x', real=True)
res = derive_envelope(x**3, x)
print('curvature        :', res.curvature.value)
print('inflection point :', res.inflection)
print('cv tangent point :', res.cv_tangent.point)   # closed form in the lower bound a


`lambdify_envelope` turns the result into a JAX closure `(x, lb, ub) -> (cv, cc)`,
and `verify_envelope` certifies soundness over many randomized boxes.


In [ ]:
relax = lambdify_envelope(res)
report = verify_envelope(relax, lambda t: t**3, domain=(-2.0, 2.0), n_boxes=400)
print('sound           :', report.sound)
print('max lower viol  :', f'{report.max_lower_violation:.2e}')
print('max upper viol  :', f'{report.max_upper_violation:.2e}')
print('mean gap (tight) :', f'{report.mean_gap:.3f}')


## 2. Transcendental atoms: numeric tangent solving

When the tangent equation has no closed form (e.g. a sigmoid), code-gen solves
the tangent point per box with a JAX bisection baked into the closure — still
`jit`/`vmap`-compatible, still sound.


In [ ]:
sig = 1 / (1 + sp.exp(-x))
res_sig = derive_envelope(sig, x)
print('curvature   :', res_sig.curvature.value)
print('closed form :', res_sig.cv_tangent.point)   # None -> numeric bisection
relax_sig = lambdify_envelope(res_sig)
rep = verify_envelope(relax_sig, lambda t: 1/(1+jax.numpy.exp(-t)), domain=(-5.0, 5.0))
print('sound       :', rep.sound)


## 3. Gas networks: the Weymouth term $f\,|f|$

Steady-state gas flow couples squared nodal pressures to flow via the Weymouth
equation $p_i^2 - p_j^2 = c\,f\,|f|$. The term $f|f| = \operatorname{sign}(f) f^2$
is concave for $f<0$ and convex for $f>0$ — a single-inflection envelope. Relaxing
it as a generic product $f \cdot |f|$ is valid but loose; the dedicated envelope
is tighter.


In [ ]:
import jax.numpy as jnp
from discopt._jax.symbolic.domains.gas import weymouth_relax
from discopt._jax.mccormick import relax_bilinear

lb, ub = -10.0, 10.0
xs = jnp.linspace(lb, ub, 201)
wgap = jax.vmap(lambda t: (lambda c: c[1]-c[0])(weymouth_relax(t, lb, ub)))(xs)
def bilinear_gap(t):
    cv, cc = relax_bilinear(t, jnp.abs(t), lb, ub, 0.0, max(abs(lb), abs(ub)))
    return cc - cv
bgap = jax.vmap(bilinear_gap)(xs)
print('mean gap  tight envelope :', f'{float(jnp.mean(wgap)):.3f}')
print('mean gap  f * |f|        :', f'{float(jnp.mean(bgap)):.3f}')


The relaxation compiler recognizes the `f * abs(f)` pattern and routes it to the
tight envelope automatically:


In [ ]:
from discopt.modeling.core import Model
from discopt._jax.relaxation_compiler import compile_relaxation
m = Model('gas'); f = m.continuous('f', lb=-10.0, ub=10.0); m.minimize(f)
relax_fn = compile_relaxation(f * abs(f), m)
cv, cc = relax_fn(jnp.array([3.0]), jnp.array([3.0]), jnp.array([-10.0]), jnp.array([10.0]))
print('f|f| at f=3 :', 3*abs(3), '  cv=%.3f cc=%.3f' % (float(cv), float(cc)))


## 4. AC optimal power flow: trigonometric angle terms

The AC power-flow equations involve $\cos(\theta_{ij})$ and $\sin(\theta_{ij})$ of
angle differences. Convex relaxations of OPF such as the QC relaxation
{cite:p}`Coffrin2016,Kroger2018` relax these over an angle box. On $(-\pi/2,\pi/2)$
the cosine is concave; the sine has a single inflection at 0.


In [ ]:
from discopt._jax.symbolic.domains.power import cos_angle_relax, sin_angle_relax
rc = verify_envelope(cos_angle_relax, jnp.cos, domain=(-1.4, 1.4))
rs = verify_envelope(sin_angle_relax, jnp.sin, domain=(-3.0, 3.0))
print('cos envelope sound :', rc.sound, ' sin envelope sound :', rs.sound)


## 5. Chemical engineering atoms

The Arrhenius rate $\exp(-E/RT)$ is convex-then-concave in temperature (inflection
at $T=E/2R$); Langmuir/Monod saturation $x/(K+x)$ is concave.


In [ ]:
from discopt._jax.symbolic.domains.chemeng import arrhenius_relax, saturating_relax
arr = arrhenius_relax(6000.0)   # E/R in kelvin
r1 = verify_envelope(arr, lambda t: jnp.exp(-6000.0/t), domain=(250.0, 900.0))
sat = saturating_relax(1.0)
r2 = verify_envelope(sat, lambda t: t/(1.0+t), domain=(0.0, 20.0))
print('arrhenius sound :', r1.sound, ' saturating sound :', r2.sound)


## 6. The atom registry

Every domain atom is catalogued and certified through one uniform soundness gate.


In [ ]:
from discopt._jax.symbolic import registry
for name, rep in registry.certify_all(n_boxes=200).items():
    print(f'{name:12s} sound={rep.sound}  mean_gap={rep.mean_gap:.3f}')


## 7. Certified learned envelopes

Guaranteed-convex networks (ICNNs) {cite:p}`Amos2017` can serve as relaxations,
but convexity alone does not guarantee the *bound* `cv <= f`. The existing
learned-relaxation wrapper clamps `cv = min(cv_pred, f)`, which restores the bound
but **destroys convexity**. A *constant* certified margin restores soundness while
preserving curvature — the structure-preserving fix.


In [ ]:
from discopt._jax.symbolic import lambdify_envelope as _le, derive_envelope as _de
from discopt._jax.symbolic.certified_learned import certify_relaxation
exact = _le(_de(x**2, x))
raw = lambda t, lb, ub: (exact(t, lb, ub)[0] + 0.7, exact(t, lb, ub)[1] - 0.7)  # convex but unsound
cert = certify_relaxation(raw, lambda t: t**2, domain=(-2.0, 2.0))
print('raw       sound=%s convexV=%.1e' % (cert.raw_report.sound, cert.raw_report.max_convexity_violation))
print('certified sound=%s convexV=%.1e' % (cert.certified_report.sound, cert.certified_report.max_convexity_violation))


## Summary

- The SymPy engine derives and certifies convex/concave envelopes; the solver
  runs only the generated JAX (`runtime.py`), keeping SymPy off the hot path.
- Single-inflection atoms use a tangent/secant construction with a closed-form or
  numeric tangent point and an automatic secant fallback on asymmetric boxes.
- Domain packs cover gas (Weymouth), power (QC trig), and chemical-engineering
  atoms, all routed through one certification gate.
- Guaranteed-convex ML can be certified into sound relaxations with a constant
  margin {cite:p}`Amos2017`.

See `design/symbolic-envelopes-plan.md` for the full roadmap.
